# Visualize MSD Task05 and nnU-Net Dataset501 Samples

This notebook checks that MSD Task05 Prostate images, binary prostate ROI labels, and the T2-only nnU-Net `Dataset501_ProstateROI_T2` conversion are aligned before training.

Dataset501 is **T2-only**. The labels are **whole-prostate ROI masks**, created by merging MSD PZ/TZ labels. They are **not tumor masks**. Visual QC is required before nnU-Net training.

In [ ]:
from pathlib import Path
import json

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"

## Project Paths

All paths are project-root-relative. Run this notebook from the repository root, or from a subdirectory inside the repository.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "AGENTS.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing AGENTS.md and data/.")


PROJECT_ROOT = find_project_root()
RAW_MSD = PROJECT_ROOT / "data/raw/public/Task05_Prostate"
BINARY_ROI = PROJECT_ROOT / "data/interim/public/msd_prostate_binary_roi"
NNUNET_DATASET = PROJECT_ROOT / "data/nnunet/nnUNet_raw/Dataset501_ProstateROI_T2"
QC_OUTPUT_DIR = PROJECT_ROOT / "outputs/figures/qc/msd_dataset501"
SAVE_QC_FIGURES = False

for path in [RAW_MSD, BINARY_ROI, NNUNET_DATASET]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw MSD: {RAW_MSD.relative_to(PROJECT_ROOT)}")
print(f"Binary ROI: {BINARY_ROI.relative_to(PROJECT_ROOT)}")
print(f"nnU-Net Dataset501: {NNUNET_DATASET.relative_to(PROJECT_ROOT)}")

## File Counts

These counts should match the expected MSD and Dataset501 structure before training.

In [ ]:
def nifti_files(path: Path) -> list[Path]:
    return sorted(
        p for p in path.glob("*.nii*")
        if not p.name.startswith("._") and (p.name.endswith(".nii") or p.name.endswith(".nii.gz"))
    )


counts = {
    "raw imagesTr": len(nifti_files(RAW_MSD / "imagesTr")),
    "raw labelsTr": len(nifti_files(RAW_MSD / "labelsTr")),
    "binary ROI labels": len(nifti_files(BINARY_ROI / "labelsTr")),
    "nnU-Net imagesTr": len(nifti_files(NNUNET_DATASET / "imagesTr")),
    "nnU-Net labelsTr": len(nifti_files(NNUNET_DATASET / "labelsTr")),
    "nnU-Net imagesTs": len(nifti_files(NNUNET_DATASET / "imagesTs")),
}

for name, count in counts.items():
    print(f"{name}: {count}")

assert counts["raw imagesTr"] == 32
assert counts["raw labelsTr"] == 32
assert counts["binary ROI labels"] == 32
assert counts["nnU-Net imagesTr"] == 32
assert counts["nnU-Net labelsTr"] == 32
assert counts["nnU-Net imagesTs"] == 16

## Helper Functions

In [ ]:
def load_nifti(path: Path):
    """Load a NIfTI image and return the nibabel image plus array data."""
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    return image, data


def choose_mask_slice(mask: np.ndarray) -> int:
    """Choose the axial slice with the largest mask area, or the center slice."""
    if mask.ndim != 3:
        raise ValueError(f"Expected 3D mask, got shape {mask.shape}")
    mask_area = (mask > 0).sum(axis=(0, 1))
    if mask_area.max() > 0:
        return int(mask_area.argmax())
    return mask.shape[2] // 2


def show_overlay(image: np.ndarray, mask: np.ndarray, title: str):
    """Show a transparent prostate ROI mask overlay on a T2 axial slice."""
    if image.ndim != 3:
        raise ValueError(f"Expected 3D image, got shape {image.shape}")
    if mask.ndim != 3:
        raise ValueError(f"Expected 3D mask, got shape {mask.shape}")
    if image.shape != mask.shape:
        raise ValueError(f"Image/mask shape mismatch: {image.shape} vs {mask.shape}")

    z = choose_mask_slice(mask)
    image_slice = image[:, :, z]
    mask_slice = mask[:, :, z]
    overlay = np.ma.masked_where(mask_slice <= 0, mask_slice)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image_slice.T, cmap="gray", origin="lower")
    ax.imshow(overlay.T, cmap="autumn", alpha=0.35, origin="lower")
    ax.set_title(f"{title} | slice {z}")
    ax.axis("off")
    fig.tight_layout()

    if SAVE_QC_FIGURES:
        QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        safe_title = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in title)
        fig.savefig(QC_OUTPUT_DIR / f"{safe_title}.png", dpi=150, bbox_inches="tight")

    plt.show()

## Visual QC Samples

The first three training cases are loaded from the MSD metadata order. For each case we inspect: raw MSD T2 plus original zone label, raw MSD T2 plus binary ROI label, and nnU-Net T2 plus nnU-Net binary label.

In [ ]:
with (RAW_MSD / "dataset.json").open("r", encoding="utf-8") as f:
    msd_metadata = json.load(f)

sample_indices = [0, 1, 2]
assert len(msd_metadata["training"]) >= len(sample_indices)

for nn_idx in sample_indices:
    entry = msd_metadata["training"][nn_idx]
    raw_image_path = RAW_MSD / entry["image"].removeprefix("./")
    raw_label_path = RAW_MSD / entry["label"].removeprefix("./")
    binary_label_path = BINARY_ROI / "labelsTr" / raw_label_path.name
    nnunet_image_path = NNUNET_DATASET / "imagesTr" / f"prostate_{nn_idx:03d}_0000.nii.gz"
    nnunet_label_path = NNUNET_DATASET / "labelsTr" / f"prostate_{nn_idx:03d}.nii.gz"

    raw_image, raw_image_data = load_nifti(raw_image_path)
    _, raw_label = load_nifti(raw_label_path)
    _, binary_label = load_nifti(binary_label_path)
    nnunet_image, nnunet_t2 = load_nifti(nnunet_image_path)
    _, nnunet_label = load_nifti(nnunet_label_path)

    assert raw_image_data.ndim == 4, raw_image_data.shape
    raw_t2 = raw_image_data[..., 0]
    assert raw_t2.shape == raw_label.shape

    binary_values = sorted(np.unique(binary_label).astype(int).tolist())
    assert set(binary_values).issubset({0, 1}), binary_values
    print(f"Case {nn_idx:03d}: {raw_image_path.name}")
    print(f"  raw 4D shape: {raw_image_data.shape}; T2 shape: {raw_t2.shape}")
    print(f"  binary ROI unique values: {binary_values}")

    assert nnunet_t2.ndim == 3, nnunet_t2.shape
    assert nnunet_t2.shape == nnunet_label.shape
    nnunet_values = sorted(np.unique(nnunet_label).astype(int).tolist())
    assert set(nnunet_values).issubset({0, 1}), nnunet_values
    print(f"  nnU-Net T2 shape: {nnunet_t2.shape}; label values: {nnunet_values}")

    show_overlay(raw_t2, raw_label, f"raw_msd_original_zone_label_{nn_idx:03d}")
    show_overlay(raw_t2, binary_label, f"raw_msd_binary_roi_{nn_idx:03d}")
    show_overlay(nnunet_t2, nnunet_label, f"nnunet_dataset501_binary_roi_{nn_idx:03d}")

## Notes

- Dataset501 contains only MSD channel `0` / T2.
- Labels are binary prostate ROI masks, not tumor masks and not cancer diagnosis labels.
- Visual QC should be reviewed before running nnU-Net planning, preprocessing, or training.
- To save PNG overlays, set `SAVE_QC_FIGURES = True`; figures will be written to `outputs/figures/qc/msd_dataset501/`.